# Final submission: 4-signal Llama ensemble, everything loaded locally

Course: Introduction to DL and GenAI Project
Competition: Smart MCQ Solver Challenge
Roll No: 22f3000114

## Inputs to attach

1. `jjinho/wikipedia-2023-07-faiss-index`
2. `jjinho/wikipedia-20230701`
3. Competition data
4. From the Models tab: `metaresearch/llama-3.1` (Transformers, 8b-instruct, V2)
5. From the Models tab: `srg9000/all-minilm-l6-v2`

The M4 adapter is pulled with kagglehub inside the notebook, no attachment needed. GPU T4 x2, WANDB secret optional. No HF_TOKEN needed, nothing downloads from the Hugging Face hub.

## The pipeline

Because of the Kaggle proxy issue with Hugging Face downloads, every model in this notebook loads from local Kaggle paths. The cross-encoder rerank stage is dropped and retrieval is bi-encoder only, which our Milestone 3 hit-rate measurement (73 percent at k=5) showed is still a usable retriever on its own.

For each question we average the letter probabilities of four signals:

1. base Llama-3.1-8B with retrieved Wikipedia context
2. base Llama-3.1-8B closed book
3. fine-tuned Llama (M4 QLoRA adapter) with context
4. fine-tuned Llama closed book

The base model brings general knowledge, the fine-tuned model has learned this competition's answer patterns from the training rows, and averaging is robust to bad retrieval since a wrong context is one vote in four. This is the Milestone 5 ensembling idea, probability averaging over diverse models.


## Install, imports and settings


In [1]:
!pip install -qU bitsandbytes peft kagglehub
!pip install -q sentence-transformers faiss-cpu wandb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.7 MB/s eta 0:00:00


In [2]:
import glob, os, gc, pickle, numpy as np, pandas as pd, torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm

print('CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
import bitsandbytes as bnb_pkg
print('bitsandbytes:', bnb_pkg.__version__)

OPTIONS = ['A', 'B', 'C', 'D', 'E']

K_ARTICLES = 10     # wikipedia articles per option query
K_FINAL    = 5      # chunks kept for the prompt
CTX_CHARS  = 4500
N_VAL      = 200

# local model paths (attached from the Kaggle Models tab)
LLAMA_DIR  = '/kaggle/input/models/metaresearch/llama-3.1/transformers/8b-instruct/2'
MINILM_DIR = '/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2'
ADAPTER_HANDLE = 'dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter'

def resolve_dir(exact, keyword):
    if os.path.isdir(exact):
        return exact
    hits = glob.glob('/kaggle/input/**/config.json', recursive=True)
    for h in hits:
        if keyword in h.lower():
            return os.path.dirname(h)
    raise FileNotFoundError(f'{keyword} model folder not found, attach it from the Models tab')

LLAMA_DIR  = resolve_dir(LLAMA_DIR, 'llama-3.1')
MINILM_DIR = resolve_dir(MINILM_DIR, 'minilm')
print('Llama :', LLAMA_DIR)
print('MiniLM:', MINILM_DIR)


CUDA available: True | GPUs: 2
bitsandbytes: 0.49.2
Llama : /kaggle/input/models/metaresearch/llama-3.1/transformers/8b-instruct/2
MiniLM: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2


## Load the competition data


In [3]:
def find_file(name):
    for pat in [f'/kaggle/input/**/{name}', f'./{name}', name]:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name + ' not found')

train = pd.read_csv(find_file('train.csv'))
test  = pd.read_csv(find_file('test.csv'))
print('train:', train.shape, '| test:', test.shape)


train: (2000, 8) | test: (500, 7)


## Load the Wikipedia FAISS index


In [4]:
WIKI_INDEX_PATH = glob.glob('/kaggle/input/**/wikipedia_202307.index', recursive=True)
idx_parquet = glob.glob('/kaggle/input/**/wiki_2023_index.parquet', recursive=True)
assert WIKI_INDEX_PATH, 'Attach jjinho/wikipedia-2023-07-faiss-index'
assert idx_parquet, 'Attach jjinho/wikipedia-20230701'
WIKI_INDEX_PATH = WIKI_INDEX_PATH[0]
WIKI_PARQUET_DIR = os.path.dirname(idx_parquet[0])

print('Loading FAISS index, takes a minute')
wiki_index = faiss.read_index(WIKI_INDEX_PATH)
print('Index vectors:', wiki_index.ntotal)
wiki_map = pd.read_parquet(idx_parquet[0])
print('columns:', list(wiki_map.columns))


Loading FAISS index, takes a minute
Index vectors: 6286775
columns: ['id', 'title', 'text', 'categories', 'context', 'file']


## Stage 1: find candidate articles (one query per option, local MiniLM)


In [5]:
bi_encoder = SentenceTransformer(MINILM_DIR, device='cuda')

def build_queries(row):
    p = str(row['prompt'])
    return [f'{p} {row[o]}' for o in OPTIONS]

val_rows  = [train.iloc[i] for i in range(N_VAL)]
test_rows = [test.iloc[i] for i in range(len(test))]
all_rows  = val_rows + test_rows

queries = []
for r in all_rows:
    queries.extend(build_queries(r))
q_emb = bi_encoder.encode(queries, batch_size=64, show_progress_bar=True)
q_emb = np.asarray(q_emb, dtype='float32')
_, art = wiki_index.search(q_emb, K_ARTICLES)
article_rows = art.reshape(len(all_rows), 5 * K_ARTICLES)
print('article_rows:', article_rows.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

article_rows: (700, 50)


## Stage 2: load article texts, checkpoint by article id


In [6]:
import pyarrow.parquet as pq

def save_ckpt(name, obj):
    with open(f'/kaggle/working/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
def load_ckpt(name):
    p = f'/kaggle/working/{name}.pkl'
    return pickle.load(open(p, 'rb')) if os.path.exists(p) else None

def chunk_article(title, text):
    parts = [p.strip() for p in str(text).split('\n') if len(p.strip()) > 80]
    if not parts:
        t = str(text)
        parts = [t[i:i+800] for i in range(0, min(len(t), 30000), 700)]
    return [f'{title}. {p}'[:1500] for p in parts]

needed = wiki_map.iloc[article_rows.flatten()].copy()
file_col = 'file' if 'file' in needed.columns else needed.columns[-1]
id_col   = 'id'   if 'id'   in needed.columns else needed.columns[0]
needed_by_file = needed.groupby(file_col)[id_col].apply(
    lambda s: set(str(x) for x in s)).to_dict()

article_chunks = load_ckpt('article_chunks') or {}
for fname, id_set in tqdm(needed_by_file.items(), desc='parquet files'):
    missing = {i for i in id_set if i not in article_chunks}
    if not missing:
        continue
    fpath = os.path.join(WIKI_PARQUET_DIR, str(fname))
    if not os.path.exists(fpath):
        hits = glob.glob(os.path.join(WIKI_PARQUET_DIR, '**', str(fname)), recursive=True)
        if not hits:
            continue
        fpath = hits[0]
    pf = pq.ParquetFile(fpath)
    tcol = 'text' if 'text' in pf.schema.names else [c for c in pf.schema.names if c not in ('id', 'title')][-1]
    cols = [c for c in ['id', 'title', tcol] if c in pf.schema.names]
    for batch in pf.iter_batches(batch_size=5000, columns=cols):
        bdf = batch.to_pandas()
        bdf['id'] = bdf['id'].astype(str)
        hit = bdf[bdf['id'].isin(missing)]
        for _, r in hit.iterrows():
            title = str(r['title']) if 'title' in bdf.columns else ''
            article_chunks[r['id']] = chunk_article(title, r[tcol])
        del bdf, hit
    del pf
    gc.collect()
    save_ckpt('article_chunks', article_chunks)

all_wanted = set(str(x) for x in needed[id_col])
cov = len(all_wanted & set(article_chunks))
print(f'coverage: {cov} of {len(all_wanted)} ({100*cov/len(all_wanted):.1f} percent)')
assert cov / len(all_wanted) > 0.95, 'coverage below 95 percent, investigate before continuing'


parquet files:   0%|          | 0/28 [00:00<?, ?it/s]

coverage: 4322 of 4322 (100.0 percent)


## Stage 3: pick the best chunks per question, bi-encoder only

The cross-encoder rerank is dropped because its weights cannot be downloaded through the current Kaggle proxy issue. The bi-encoder ranks each question's pooled chunks by cosine similarity and the top 5 become the context. Milestone 3 measured a 73 percent top-5 hit rate for this retriever on its own, so it remains a usable, if less precise, retrieval stage.


In [7]:
contexts = load_ckpt('contexts_v4')
if contexts is None:
    contexts = []
    for qi in tqdm(range(len(all_rows)), desc='retrieve'):
        row = all_rows[qi]
        seen, chunks = set(), []
        for art_row in article_rows[qi]:
            if art_row in seen:
                continue
            seen.add(art_row)
            aid = str(wiki_map.iloc[art_row][id_col])
            chunks.extend(article_chunks.get(aid, []))
        if not chunks:
            contexts.append(''); continue
        q_text = str(row['prompt']) + ' ' + ' '.join(str(row[o]) for o in OPTIONS)
        c_emb = bi_encoder.encode(chunks, batch_size=128, show_progress_bar=False)
        qv = bi_encoder.encode([q_text], show_progress_bar=False)[0]
        sims = c_emb @ qv / (np.linalg.norm(c_emb, axis=1) * np.linalg.norm(qv) + 1e-9)
        top = np.argsort(-sims)[:K_FINAL]
        contexts.append(' '.join(chunks[i] for i in top)[:CTX_CHARS])
    save_ckpt('contexts_v4', contexts)

print('contexts ready:', len(contexts), '| empty:', sum(1 for c in contexts if not c))


retrieve:   0%|          | 0/700 [00:00<?, ?it/s]

contexts ready: 700 | empty: 0


## Free retrieval models before loading Llama


In [8]:
del bi_encoder, wiki_index, wiki_map, q_emb, article_chunks
gc.collect()
torch.cuda.empty_cache()
for d in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(d)
    print(f'GPU {d}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')


GPU 0: 15.5 GB free of 15.6 GB
GPU 1: 15.5 GB free of 15.6 GB


## Load Llama from the local Kaggle Models path, plus the M4 adapter via kagglehub

Nothing here touches the Hugging Face hub. The base model loads from the attached Kaggle Models folder, 4-bit NF4 with float16 compute for the T4s, one model sharded across both GPUs. The M4 adapter downloads through kagglehub, Kaggle's own infrastructure. If the adapter download fails for any reason the notebook continues as a 2-signal base-only ensemble instead of stopping.


In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no native bfloat16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA_DIR, quantization_config=bnb_config, device_map='auto',
    low_cpu_mem_usage=True)
model.eval()
print('Base model loaded from disk.')

HAS_ADAPTER = False
try:
    import kagglehub
    from peft import PeftModel
    adapter_dir = kagglehub.model_download(ADAPTER_HANDLE)
    model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    HAS_ADAPTER = True
    print('Fine-tuned adapter loaded from', adapter_dir, '-> 4-signal ensemble')
except Exception as e:
    print('Adapter not loaded (', e, ') -> 2-signal base-only ensemble')


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base model loaded from disk.
Fine-tuned adapter loaded from /kaggle/input/models/dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter/1 -> 4-signal ensemble


## Logit extraction and the ensemble

One forward pass per signal, read the next-token logits for the five letter tokens (max over the bare and leading-space token ids), softmax to probabilities, average across the signals.


In [10]:
letter_ids = {}
for o in OPTIONS:
    ids = set()
    ids.add(tokenizer.encode(o, add_special_tokens=False)[-1])
    ids.add(tokenizer.encode(' ' + o, add_special_tokens=False)[-1])
    letter_ids[o] = list(ids)
print('letter token ids:', letter_ids)

def build_prompt(row, context):
    q = str(row['prompt'])
    opts = '\n'.join(f'{o}. {row[o]}' for o in OPTIONS)
    if context:
        return f'Context:\n{context}\n\nQuestion: {q}\n\nOptions:\n{opts}\n\nAnswer:'
    return f'Question: {q}\n\nOptions:\n{opts}\n\nAnswer:'

@torch.no_grad()
def letter_probs(row, context, use_adapter):
    input_ids = tokenizer(build_prompt(row, context), return_tensors='pt').input_ids.to(model.device)
    if HAS_ADAPTER and not use_adapter:
        with model.disable_adapter():
            logits = model(input_ids).logits[0, -1]
    else:
        logits = model(input_ids).logits[0, -1]
    raw = torch.tensor([max(logits[i].item() for i in letter_ids[o]) for o in OPTIONS])
    return torch.softmax(raw, dim=0).numpy()

def ensemble_probs(row, context):
    sigs = [letter_probs(row, context, False),
            letter_probs(row, '', False)]
    if HAS_ADAPTER:
        sigs.append(letter_probs(row, context, True))
        sigs.append(letter_probs(row, '', True))
    return np.mean(sigs, axis=0)

def top3(p):
    return [OPTIONS[i] for i in np.argsort(-p)[:3]]

print('signals per question:', 4 if HAS_ADAPTER else 2)
print('demo row 0 ->', top3(ensemble_probs(train.iloc[0], contexts[0])), '| true', train.iloc[0]['answer'])


letter token ids: {'A': [32, 362], 'B': [33, 426], 'C': [34, 356], 'D': [35, 423], 'E': [36, 469]}
signals per question: 4
demo row 0 -> ['B', 'A', 'D'] | true B


## Validate on the held-out 200 rows

Prints each signal's MAP@3 and the ensemble's. Note that the base model's numbers on train rows may be optimistic if it saw similar questions in pretraining, the tuned and ensemble numbers on these held-out rows are the more reliable guide.


In [11]:
def map_at_3(predictions, labels):
    weights = [1.0, 1/2, 1/3]
    total = 0.0
    for preds, true in zip(predictions, labels):
        s = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                s = weights[i]; break
        total += s
    return total / len(labels)

val_true = [r['answer'] for r in val_rows]
val_ens_probs = []
sig_names = ['base+RAG', 'base+closed'] + (['tuned+RAG', 'tuned+closed'] if HAS_ADAPTER else [])
sig_preds = {n: [] for n in sig_names}

for i in tqdm(range(N_VAL), desc='validate'):
    row, ctx = val_rows[i], contexts[i]
    s = [letter_probs(row, ctx, False), letter_probs(row, '', False)]
    if HAS_ADAPTER:
        s += [letter_probs(row, ctx, True), letter_probs(row, '', True)]
    for n, p in zip(sig_names, s):
        sig_preds[n].append(top3(p))
    val_ens_probs.append(np.mean(s, axis=0))

for n in sig_names:
    print(f'{n:14s} MAP@3: {map_at_3(sig_preds[n], val_true):.4f}')
m_ens = map_at_3([top3(p) for p in val_ens_probs], val_true)
print(f'{"ENSEMBLE":14s} MAP@3: {m_ens:.4f}')
np.save('/kaggle/working/llama_val_ens_probs.npy', np.array(val_ens_probs))


validate:   0%|          | 0/200 [00:00<?, ?it/s]

base+RAG       MAP@3: 0.8908
base+closed    MAP@3: 0.8150
tuned+RAG      MAP@3: 1.0000
tuned+closed   MAP@3: 0.9842
ENSEMBLE       MAP@3: 0.9925


## Predict the test set and write submission.csv


In [12]:
test_probs, test_preds = [], []
for i in tqdm(range(len(test)), desc='test'):
    p = ensemble_probs(test.iloc[i], contexts[N_VAL + i])
    test_probs.append(p)
    test_preds.append(' '.join(top3(p)))

np.save('/kaggle/working/llama_test_ens_probs.npy', np.array(test_probs))
submission = pd.DataFrame({'id': test['id'], 'prediction': test_preds})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print('Wrote submission.csv with', len(submission), 'rows')


test:   0%|          | 0/500 [00:00<?, ?it/s]

   id prediction
0   1      D A E
1   2      B A D
2   3      B C A
3   4      E A C
4   5      C A B
Wrote submission.csv with 500 rows


## Log to W&B


In [13]:
try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB'))
    run = wandb.init(entity='22f3000114-dl-genai-project', project='dl-genai-project',
                     name='llama-ensemble-v4' + ('-4sig' if HAS_ADAPTER else '-2sig'),
                     group='leaderboard',
                     config={'base': 'llama-3.1-8b-instruct (local)', 'adapter': HAS_ADAPTER,
                             'retrieval': 'bi-encoder only (local MiniLM)',
                             'k_articles': K_ARTICLES, 'k_final': K_FINAL}, reinit=True)
    log = {f'val_map3_{n}': map_at_3(sig_preds[n], val_true) for n in sig_names}
    log['val_map3_ensemble'] = m_ens
    run.log(log)
    run.finish()
    print('Logged to W&B.')
except Exception as e:
    print('W&B logging skipped:', e)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run 2fmumbox
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260714_200645-2fmumbox
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run llama-ensemble-v4-4sig
wandb: ⭐️ View project at https://wandb.ai/22f3000114-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/22f30001

Logged to W&B.


## Notes

Fill in after running.

Record each signal's validation MAP@3, the ensemble's, and the leaderboard score, and compare against the earlier single-model runs (closed book 0.66251, gated RAG 0.6729). The key viva points from this notebook: the ensemble is probability averaging over diverse models (Milestone 5), the tuned signals come from the Milestone 4 QLoRA adapter, retrieval is the Milestone 3 pipeline with the rerank stage dropped for infrastructure reasons and the hit-rate evidence for why bi-encoder-only retrieval remains defensible, and logit extraction is the Day 9 technique. Every model loads from Kaggle-local paths because of the documented Kaggle proxy issue with Hugging Face downloads, which is itself worth one line in the report as an engineering constraint that shaped the final design.
